# Create an FMI 2 container FMU

This container connects the FMI 2 Controller, Drive, and Stimuli FMUs and exposes the drive speed `w`.

In [ ]:
from pathlib import Path
from uuid import uuid4

from fmpy import plot_result, read_model_description, simulate_fmu
from fmpy.container_fmu.cli import create_container_fmu
from fmpy.container_fmu.config import Component, Configuration, Connection
from fmpy.model_description import CoSimulation, ModelDescription, ModelVariable, Unknown


fmus_dir = Path('fmus')
container_filename = fmus_dir / 'DriveSystem_Container_FMU_2.fmu'
container_unzipdir = fmus_dir / 'DriveSystem_Container_FMU_2'


def get_variable(model_description, name):
    variable = next(variable for variable in model_description.modelVariables if variable.name == name)
    if variable.type == 'Real':
        variable.type = 'Float64'
    return variable


def component(name, filename):
    return Component(
        name=name,
        filename=filename,
        modelDescription=read_model_description(filename),
    )


controller = component('Controller', fmus_dir / 'Controller_FMU_2.fmu')
drive = component('Drive', fmus_dir / 'Drive_FMU_2.fmu')
stimuli = component('Stimuli', fmus_dir / 'Stimuli_FMU_2.fmu')

controller_w_desired = get_variable(controller.modelDescription, 'w_desired')
controller_v = get_variable(controller.modelDescription, 'V')
controller_w = get_variable(controller.modelDescription, 'w')
drive_v = get_variable(drive.modelDescription, 'V')
drive_w = get_variable(drive.modelDescription, 'w')
drive_load_torque = get_variable(drive.modelDescription, 'LoadTorque_Nm')
stimuli_w_desired = get_variable(stimuli.modelDescription, 'w_desired')
stimuli_load_torque = get_variable(stimuli.modelDescription, 'LoadTorque_Nm')

container_time = ModelVariable(
    name='time',
    type='Real',
    causality='independent',
    variability='continuous',
)
container_w = ModelVariable(
    name='w',
    type='Real',
    causality='output',
    variability='continuous',
    initial='calculated',
)
container_model_description = ModelDescription(
    fmiVersion='2.0',
    modelName='DriveSystem_Container_FMU_2',
    instantiationToken=str(uuid4()),
    coSimulation=CoSimulation(
        canHandleVariableCommunicationStepSize=True,
        fixedInternalStepSize=0.001,
    ),
    modelVariables=[container_time, container_w],
    outputs=[Unknown(index=2, variable=container_w)],
    initialUnknowns=[Unknown(index=2, variable=container_w)],
)
container_configuration = Configuration(
    parallelDoStep=False,
    modelDescription=container_model_description,
    components=[controller, drive, stimuli],
    connections=[
        Connection(stimuli, [stimuli_w_desired], controller, [controller_w_desired]),
        Connection(stimuli, [stimuli_load_torque], drive, [drive_load_torque]),
        Connection(controller, [controller_v], drive, [drive_v]),
        Connection(drive, [drive_w], controller, [controller_w]),
    ],
    variableMappings={container_w: [(drive, drive_w)]},
)

container_unzipdir.mkdir(exist_ok=True)
if container_filename.exists():
    container_filename.unlink()

create_container_fmu(container_configuration, container_unzipdir, container_filename)

container_result = simulate_fmu(
    container_filename,
    fmi_type='CoSimulation',
    output=['w'],
    stop_time=1.0,
)

plot_result(container_result)